<a href="https://colab.research.google.com/github/laisalegri/curso-fiap-agentes-autonomos-ia/blob/main/FIAP_hands_on_2_agente_atendimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install langchain -q
!pip install langchain-core -q
!pip install langchain-community -q
!pip install langchain-ollama -q
!pip install langgraph -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [10]:
!pip install -U langchain-ollama

In [11]:
!pgrep -af ollama
!curl -s http://127.0.0.1:11434/api/tags

In [13]:
# 1. Instala o zstd e depois instala o Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Inicia o servidor do Ollama em background
!nohup ollama serve > /dev/null 2>&1 &

# 3. Faz o download do modelo
!ollama pull gpt-oss:20b

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,044 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security

In [15]:
from langchain_community.llms import Ollama

def processamento_resposta(pergunta):
  # Initialize the Ollama LLM
  llm = Ollama(model="gpt-oss:20b",num_predict=1024)

  # Send a prompt
  resposta = llm.invoke(pergunta)
  return resposta

processamento_resposta("Diga: Hello World!")


'Hello World!'

In [17]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama

In [20]:
# Definir o schema do estado
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Inicializar o modelo
llm = ChatOllama(model="gpt-oss:20b", temperature=0)

# Nó de entrada
def entrada_usuario(state: State) -> State:
     pergunta = input("Pergunta: ")
     if isinstance(pergunta, str) and pergunta.strip():
         return {"messages": [HumanMessage(content=pergunta)]}
     else:
         raise ValueError("A pergunta deve ser uma string não vazia.")

# Nó de processamento usando ChatOllama
def processar_solicitacao(state: State) -> State:
     last_message = state["messages"][-1]

     if isinstance(last_message, HumanMessage) and last_message.content.strip():
         print(f"DEBUG: Processando pergunta: {last_message.content}")
         try:
             resposta = llm.invoke([last_message]).content
             print(f"DEBUG: Resposta gerada: {resposta}")
             return {"messages": [AIMessage(content=resposta)]}
         except Exception as e:
             print(f"DEBUG: Erro ao chamar o LLM: {e}")
             return {"messages": [AIMessage(content=f"Desculpe, ocorreu um erro: {e}")]}

     else:
         raise ValueError("A última mensagem no estado não é uma HumanMessage válida.")

# Nó de saída
def saida_resposta(state: State) -> State:
     last_message = state["messages"][-1]
     if hasattr(last_message, 'content'):
         print(f"Resposta: {last_message.content}")
     else:
         print(f"Resposta: {last_message}")
     return state

# Criar o grafo de estados
grafo = StateGraph(State)
grafo.add_node("entrada", entrada_usuario)
grafo.add_node("processamento", processar_solicitacao)
grafo.add_node("saida", saida_resposta)

grafo.add_edge(START, "entrada")
grafo.add_edge("entrada", "processamento")
grafo.add_edge("processamento", "saida")
grafo.add_edge("saida", END)

compiled = grafo.compile()

# Loop de interação
print("Iniciando a interação...")
while True:
     try:
         compiled.invoke({"messages": []})
         continuar = input("Deseja fazer outra pergunta? (sim/não): ")
         if continuar.lower() != "sim":
             break
     except ValueError as ve:
         print(f"Erro de entrada: {ve}")
     except KeyboardInterrupt:
         print("\nInteração encerrada pelo usuário.")
         break
     except Exception as e:
         print(f"Ocorreu um erro inesperado: {e}")
         break

print("Interação encerrada.")

Iniciando a interação...
Pergunta: Oi boa tarde tudo bem?
DEBUG: Processando pergunta: Oi boa tarde tudo bem?
DEBUG: Resposta gerada: Oi! Boa tarde! Tudo bem por aqui. Como posso ajudar você hoje?
Resposta: Oi! Boa tarde! Tudo bem por aqui. Como posso ajudar você hoje?
Deseja fazer outra pergunta? (sim/não): sim
Pergunta: Quais são as 7 maravilhas do mundo?
DEBUG: Processando pergunta: Quais são as 7 maravilhas do mundo?
DEBUG: Resposta gerada: **Quais são as 7 maravilhas do mundo?**

A expressão “7 maravilhas do mundo” pode se referir a diferentes listas que foram criadas ao longo da história. Abaixo estão as duas mais conhecidas:

---

## 1. As 7 Maravilhas do Mundo Antigo  
(Conjunto de monumentos que os antigos gregos e romanos consideravam as mais impressionantes do seu tempo)

| # | Maravilha | Local | Status atual |
|---|-----------|-------|--------------|
| 1 | **Grande Pirâmide de Gizé** | Egito | **Presente** – a única das sete que ainda existe. |
| 2 | **Jardins Suspensos da